In [7]:
# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal
import warnings

# warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
sys.path.append(root_path)

In [8]:
from core.backtesting import BacktestingEngine

backtesting = BacktestingEngine(root_path=root_path, load_cached_data=True)

In [9]:
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
import datetime
from decimal import Decimal, getcontext
from controllers.directional_trading.pz_scalper import PZScalperControllerConfig
getcontext().prec = 4  # set desired precision

# Controller configuration
connector_name = "binance_perpetual"
trading_pair = "BTC-USDT"
interval = "15m"
backtesting_resolution = "1m"

# Don't matter
cooldown_time = 1 #60 * 15
take_profit = 5 # 100%, -> Disable Take profit, let the trailing do it's job
stop_loss = 5
# trailing_stop_activation_price = 1
# trailing_stop_trailing_delta = 0.05

# General
total_amount_quote: int = 1000
max_executors_per_side: int = 1


# Indicator Values
ema_fast: int = 30
ema_slow: int = 70
srsi_smoothing: int = 4
srsi_length: int = 12
rsi_ma_length: int  = 6
hma_diff_ma_length: int = 6
hma_fast: int = 10
hma_slow: int = 40
natr_length: int = 13

# Triple Barrier
time_limit: int = 900
tp_natr_factor = 0.75
sl_natr_factor = 2.5
# ts_activation_natr_factor = 1.0
# ts_delta_natr_factor = 0.75
###


# Creating the instance of the configuration and the controller
config = PZScalperControllerConfig(
    connector_name=connector_name,
    trading_pair=trading_pair,
    interval=interval,
    take_profit=Decimal(take_profit),
    stop_loss=Decimal(stop_loss),
    # trailing_stop=TrailingStop(activation_price=Decimal(trailing_stop_activation_price), trailing_delta=Decimal(trailing_stop_trailing_delta)),
    total_amount_quote=Decimal(total_amount_quote),
    time_limit=time_limit,
    max_executors_per_side=max_executors_per_side,
    cooldown_time=cooldown_time,
    natr_length = natr_length,
    sl_natr_factor=sl_natr_factor,
    # ts_activation_natr_factor = ts_activation_natr_factor,
    # ts_delta_natr_factor = ts_delta_natr_factor,
    tp_natr_factor=tp_natr_factor,
    ema_fast=ema_fast,
    ema_slow=ema_slow,
    rsi_ma_length=rsi_ma_length,
    srsi_length=srsi_length,
    srsi_smoothing=srsi_smoothing,
    hma_diff_ma_length=hma_diff_ma_length,
    hma_fast=hma_fast,
    hma_slow=hma_slow
)

In [10]:
# Running the backtesting this will output a backtesting result object that has built in methods to visualize the results

start = int(datetime.datetime(2025, 2, 25).timestamp())
end = int(datetime.datetime(2025, 3, 30).timestamp())
maker_fee = Decimal(0.0002)
taker_fee = Decimal(0.0006)
# Worst case, MKT entry and Stop via MKT order
trade_cost: Decimal = Decimal(2) * Decimal(taker_fee)
#+ maker_fee

backtesting_result = await backtesting.run_backtesting(config, start, end, backtesting_resolution, trade_cost=float(trade_cost))

2025-04-12 17:03:16,533 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x76d5ebe12a10>
2025-04-12 17:03:16,534 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x76d61468b640>, 17990.914288279)])']
connector: <aiohttp.connector.TCPConnector object at 0x76d5ebe129b0>


2025-04-12 17:03:20,763 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x76d5ebe12a40>
2025-04-12 17:03:20,764 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x76d608d4f400>, 17995.143754218)])']
connector: <aiohttp.connector.TCPConnector object at 0x76d5ebe12ef0>


In [11]:
import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# Let's see what is inside the backtesting results
print(backtesting_result.get_results_summary())
fig = backtesting_result.get_backtesting_figure()
# Add EMAs
candles_df = backtesting_result.processed_data

# fast_key = f"HMA_{hma_fast}"
# slow_key = f"HMA_{hma_slow}"


# fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[fast_key],
#                          line=dict(color='#00FFFF', width=2),
#                          name='Fast HMA'))
# fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[slow_key],
#                          line=dict(color='#FFFF00', width=2),
#                          name='Slow HMA'))
# fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[f"EMA_{ema_fast}"],
#                          line=dict(color='#FFFFFF', width=2),
#                          name='Fast EMA'))
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[f"EMA_{ema_slow}"],
                         line=dict(color='#FFFFFF', width=2),
                         name='Slow EMA'))



Net PNL: $85.52 (8.55%) | Max Drawdown: $-14.54 (-1.45%)
Total Volume ($): 150000.00 | Sharpe Ratio: 2.85 | Profit Factor: 2.09
Total Executors: 75 | Accuracy Long: 0.64 | Accuracy Short: 0.51
Close Types: Take Profit: 21 | Stop Loss: 0 | Time Limit: 54 |
             Trailing Stop: 0 | Early Stop: 0

